In [17]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [18]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [38]:
import requests

CITIES = {
    "krakow": (50.0614, 19.9366),
    "london": (51.5074, -0.1278),
    "new york": (40.7128, -74.0060),
    "madrid": (40.4168, -3.7038),
    "tokyo": (35.6762, 139.6503),
}


def get_weather(city):
    coords = CITIES.get(city.lower().strip())
    if coords is None:
        return {"error": f"No weather data for '{city}'. Supported cities: {', '.join(CITIES)}"}

    latitude, longitude = coords
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code,wind_speed_10m",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weather_code",
        "timezone": "auto",
        "forecast_days": 2,
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    return data
    
print(get_weather("london"))


{'latitude': 51.51147, 'longitude': -0.13078308, 'generationtime_ms': 0.1850128173828125, 'utc_offset_seconds': 0, 'timezone': 'Europe/London', 'timezone_abbreviation': 'GMT', 'elevation': 16.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-11-02T23:30', 'interval': 900, 'temperature_2m': 9.5, 'weather_code': 2, 'wind_speed_10m': 15.5}, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm', 'weather_code': 'wmo code'}, 'daily': {'time': ['2026-11-02', '2026-11-03'], 'temperature_2m_max': [12.1, 12.5], 'temperature_2m_min': [8.4, 8.3], 'precipitation_sum': [0.3, 0.4], 'weather_code': [51, 51]}}


In [32]:
weather_function = {
    "name": "get_weather",
    "description": "Get the current weather and tomorrow's forecast for a given city. Call this whenever the user asks about the weather.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city, e.g. Krakow or London",
            },
        },
        "required": ["city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": weather_function}]


In [33]:
system_prompt = """
You are a weather forecast assistant who delivers the weather in the style of a Dave Chappelle stand-up bit: sharp, observational, conversational, a bit sarcastic, with timing and callbacks.

Rules:
- ALWAYS use the get_weather tool to get the weather. Never guess or invent numbers.
- Cover both the current weather (today) and tomorrow's forecast.
- Include the real numbers (temperature, rain, wind) so the forecast is actually useful, and make the jokes around them.
- Keep it short: 4-6 sentences.
- Keep it light. No slurs, no attacks on groups of people. Joke about the weather, the city and everyday life, not about people's identity.
- If the user does not mention a city, ask which one.
- If the tool returns an error, say you have no weather for that city and list the available ones, with a joke.
- Reply in the same language the user writes in.
"""


In [34]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_weather":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            weather = get_weather(city)
            responses.append({
                "role": "tool",
                "content": json.dumps(weather),
                "tool_call_id": tool_call.id,
            })
    return responses


In [35]:
openai = OpenAI()
import json


def chat(user_message):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    response = openai.chat.completions.create(model="gpt-5-nano", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model="gpt-5-nano", messages=messages, tools=tools)

    return response.choices[0].message.content


In [ ]:
print(chat("weather for Krakow"))


W Krakowie teraz mamy około 5,5°C, a wiatr wieje około 10,4 km/h, więc warto mieć kaptur. Dzisiaj jest pochmurnie i sypnie drobnym deszczem – opady to 0,2 mm. Dzisiejszy dzień to maksymalnie 6,6°C i minimum 5,1°C. Jutro zrobi się nieco cieplej, maks 8,1°C, min 4,4°C, a opady to 0,1 mm. Podsumowując: kawa, parasol i Kraków – pogoda ma swój własny stand-up, my tylko kupujemy bilet.
